# All 10 GCP Agentic Patterns — Full Comparison

> Based on: https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system

This notebook is a quick reference for all 10 patterns covered across Day 1 and Day 2.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

## Pattern Overview Table

| # | Pattern | Type | Orchestration | Latency | Cost | Complexity | Best For |
|---|---------|------|--------------|---------|------|-----------|----------|
| 1 | Single Agent | Single | None | Low | Low | Low | Simple multi-step tasks, prototypes |
| 2 | Sequential | Multi | Hardcoded | Medium | Low | Low | ETL, content pipelines, ordered steps |
| 3 | Parallel | Multi | Hardcoded | Low | Medium | Low | Multi-perspective analysis, fan-out |
| 4 | Loop | Multi | Hardcoded | Variable | Variable | Medium | Polling, monitoring, retry loops |
| 5 | Review & Critique | Multi | Hardcoded | High | Medium | Medium | Quality gates, code review, safety |
| 6 | Iterative Refinement | Multi | Hardcoded | High | Medium | Medium | Complex writing, progressive improvement |
| 7 | Coordinator | Multi | AI model | Medium | Medium | Medium | Diverse inputs needing smart routing |
| 8 | Hierarchical | Multi | AI model (multi-level) | High | High | High | Complex ambiguous research, planning |
| 9 | Swarm | Multi | None (emergent) | Highest | Highest | Highest | Creative debate, strategy, consensus |
| 10 | ReAct | Single/Multi | AI model (iterative) | Medium-High | Medium | Medium | Dynamic tasks, open-ended research |
| 11 | Human-in-Loop | Any | Human + AI | Highest | Variable | High | High-stakes, compliance, safety-critical |
| 12 | Custom Logic | Any | Python code | Variable | Variable | High | Unique workflows, mixed patterns |

## Agno Framework Mapping

| Pattern | Agno Primitive |
|---------|---------------|
| Single Agent | `Agent(...)` |
| Sequential | `Team(mode='sequential')` or `Workflow` |
| Parallel | `Team(mode='parallel')` or `asyncio.gather` |
| Loop | `Workflow` with while loop + exit condition |
| Review & Critique | `Workflow` with generator + critic agents |
| Iterative Refinement | `Workflow` with quality score threshold |
| Coordinator | `Team(mode='coordinate')` |
| Hierarchical | Nested `Team` of `Team`s with `mode='coordinate'` |
| Swarm | `Team(mode='collaborate')` |
| ReAct | `Agent(show_tool_calls=True)` — default Agno behavior |
| Human-in-Loop | `Workflow` + `input()` or webhook callback |
| Custom Logic | Python `async def` + multiple `Agent.arun()` calls |

## Decision Flowchart

Use this to pick the right pattern:

```
START: What is your workflow type?
│
├─ DETERMINISTIC (predictable steps):
│   ├─ Fixed linear order → Sequential Pattern
│   ├─ Independent sub-tasks in parallel → Parallel Pattern
│   └─ Iterative improvement needed:
│       ├─ Need quality gate (generator + critic) → Review & Critique
│       └─ Progressive refinement → Iterative Refinement
│
├─ DYNAMIC ORCHESTRATION (AI decides the flow):
│   ├─ Simple multi-step + tools, prototype → Single Agent
│   ├─ Route diverse inputs to specialists → Coordinator
│   ├─ Complex ambiguous multi-domain task → Hierarchical
│   └─ Collaborative debate across experts → Swarm
│
├─ ITERATIVE:
│   ├─ Dynamic plan-act-observe cycle → ReAct
│   ├─ Repeat until exit condition → Loop
│   └─ Need quality gate in loop → Review & Critique
│
└─ SPECIAL REQUIREMENTS:
    ├─ Human oversight required → Human-in-the-Loop
    └─ Complex branching, unique logic → Custom Logic
```

## Live Pattern Advisor
Ask the advisor which pattern fits your use case.

In [2]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from pydantic import BaseModel, Field
from typing import Optional, List

class PatternChoice(BaseModel):
    pattern: str = Field(..., description="One of the 12 GCP patterns")
    reasoning: str = Field(..., description="Why this pattern fits")
    agno_code: str = Field(..., description="Agno framework skeleton code")
    tradeoffs: List[str] = Field(..., description="Key trade-offs to consider")
    gcp_url: str = Field(..., description="Link to GCP documentation")

advisor = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    description="Expert in Google Cloud's 12 agentic AI patterns.",
    output_schema=PatternChoice,  # Agno 2.x: was response_model= in older versions
    structured_outputs=True,
    instructions=[
        "Match the use case to one of the 12 GCP patterns.",
        "Recommend the SIMPLEST pattern that works — don't over-engineer.",
        "Provide real Agno framework code in the agno_code field.",
        "GCP base URL: https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system",
    ],
)

# Try your own use case here!
use_cases = [
    "I want to build a content moderation system that checks text for hate speech, spam, and misinformation simultaneously, then blocks if any check fails.",
    "I need an agent that keeps trying to book a flight until it finds one under $500, checking every hour.",
    "I want to build a medical diagnosis assistant that recommends treatment but a doctor must approve before sending to patient.",
]

for use_case in use_cases:
    print(f"\n{'='*60}")
    print(f"Use Case: {use_case[:70]}...")
    print('='*60)
    
    result = advisor.run(f"What GCP agentic pattern fits this? {use_case}")
    choice = result.content
    
    print(f"\n✅ Recommended: {choice.pattern}")
    print(f"   Why: {choice.reasoning}")
    print(f"   Docs: {choice.gcp_url}")
    print(f"\n   Code skeleton:")
    print(choice.agno_code)
    print(f"\n   Trade-offs:")
    for t in choice.tradeoffs:
        print(f"   • {t}")


Use Case: I want to build a content moderation system that checks text for hate ...

✅ Recommended: Agent with Tools
   Why: The 'Agent with Tools' pattern is suitable because you need to perform multiple distinct checks (hate speech, spam, misinformation) on the input concurrently and then make a decision based on the combined outcomes. Each check can be considered a 'tool' that the main agent invokes. The agent orchestrates the calls to these tools and aggregates their results to determine if the content should be blocked.
   Docs: https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#agent-tools

   Code skeleton:
{
  "agent_name": "ContentModeratorAgent",
  "description": "Agent that moderates text content for compliance.",
  "tools": [
    {
      "tool_name": "HateSpeechDetector",
      "description": "Detects hate speech in text.",
      "type": "function",
      "code": null
    },
    {
      "tool_name": "SpamDetector",
      "description": "Dete

## Summary: Week 12 Learning Journey

| Day | Topic | Files |
|-----|-------|-------|
| Day 1 | Agno intro + Single Agent + Tools | `0_agno_intro.ipynb` |
| Day 1 | Single Agent Pattern (Pattern 1) | `1_single_agent.ipynb` |
| Day 1 | Tools — custom + built-in + toolkits | `2_tools.ipynb` |
| Day 1 | Structured output with Pydantic | `3_structured_output.py` |
| Day 1 | Sequential (Pattern 2) + Parallel (Pattern 3) | `4_sequential_parallel.ipynb` |
| Day 1 | ReAct Pattern (Pattern 10) | `5_react_agent.py` |
| Day 1 | Agno OS — single agent playground | `6_agno_os.py` |
| Day 2 | Coordinator Pattern (Pattern 7) | `0_coordinator.ipynb` |
| Day 2 | Hierarchical Pattern (Pattern 8) | `1_hierarchical.ipynb` |
| Day 2 | Swarm Pattern (Pattern 9) | `2_swarm_pattern.ipynb` |
| Day 2 | Loop + Review-Critique + Iterative (Patterns 4-6) | `3_review_critique_loop.ipynb` |
| Day 2 | Human-in-the-Loop Pattern (Pattern 11) | `4_human_in_loop.py` |
| Day 2 | Custom Logic Pattern (Pattern 12) | `5_custom_logic.py` |
| Day 2 | Agno OS — multi-agent playground | `6_agno_os_multiagent.py` |
| Day 2 | Full comparison + decision guide | `7_pattern_comparison.ipynb` |